In [4]:
# ==== Parameters ====
INPUT_PATH = "2023-01-13Data-exportOervondstchecker.csv" 
CONTEXT_OUT = "oervondstchecker_context.csv"
IMAGES_OUT = "oervondstchecker_images.csv"


In [5]:
import pandas as pd

data = pd.read_csv(INPUT_PATH, sep=';', encoding="latin1")
data

,date,name,description,latitude,longitude,type_code,category,minimum period,maximum period,creator,reviewer,reviewer_notes,images
0,2023-01-12,Knipkies hond/wolf?,Vandaag gevonden op de zandmotor. Plek kaart k...,51.968006,3.973275,palaeontological,Mammals,n.C. 0 - v.C. 10.000 | Recent,n.C. 0 - v.C. 10.000 | Recent,Dimar,Charlie Schouwenburg,<p>Geen Maasvlakte</p>\n<p>Dit is een premlaar...,https://www.oervondstchecker.nl/images/large/4...
1,2023-01-12,Rond hol botfragment,"Niet te determineren , maar waar moet ik zon ...",51.948593,3.968725,palaeontological,Mammals,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 65.000 - v.C. 120.000 | Vroeg Weichselien,Geert,Hansjorg Ahrens,<p>Een schacht van een middenhands- of voetsbe...,https://www.oervondstchecker.nl/images/large/2...
2,2023-01-12,Stukje kroon,Zo te zien een fragment van een kroon\nNog te ...,51.945522,3.969669,palaeontological,Mammals,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 65.000 - v.C. 120.000 | Vroeg Weichselien,Geert,Charlie Schouwenburg,"<p>Dat is een moeilijke. Mijn eerste indruk, g...",https://www.oervondstchecker.nl/images/large/0...
3,2023-01-12,Redelijk compleet bot ?,Volgens mij een redelijk compleet bot\nWeet u ...,51.935310,3.977137,palaeontological,Mammals,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,Geert,Charlie Schouwenburg,"<p>Dit is een frament van de derde teenkoot, w...",https://www.oervondstchecker.nl/images/large/3...
4,2023-01-12,Botfragment,Helaas behoorlijk beschadigd maar wellicht doo...,51.928749,3.980312,palaeontological,Mammals,n.C. 0 - v.C. 10.000 | Recent,n.C. 0 - v.C. 10.000 | Recent,Geert@gmail.com,Charlie Schouwenburg,<p>Ondanks alle vormen kan ik het toch niet de...,https://www.oervondstchecker.nl/images/large/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19834,2014-01-24,Geweidelen?,Mogelijk delen van een gewei?,51.960930,3.961022,palaeontological,Mammals,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,Johan Grootveld,Charlie Schouwenburg,<p>Ik denk in alledrie de gevallen niet aan ee...,https://www.oervondstchecker.nl/images/large/5...
19835,2014-01-24,Harpoenpunt,Het lijkt op een harpoenpunt,51.939095,3.972733,archaeological,Tools,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 5.000 - v.C. 15.000 | Holoceen,Walter,Dimitri Schiltmans,<p><span>Alweer een mooi exemplaar van uit bee...,https://www.oervondstchecker.nl/images/large/5...
19836,2014-01-24,Schedelfragment,Unieke vondst van een schedelfragment,51.976078,3.972673,archaeological,Human,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 5.000 - v.C. 15.000 | Holoceen,Walter,Hansjorg Ahrens,<p>Het betreft een deel van de rechterhelft va...,https://www.oervondstchecker.nl/images/large/5...
19837,2014-01-24,Tand van witte haai,Vanmorgen toch weer een topvondst gedaan op de...,51.956669,3.962030,palaeontological,NaN,NaN,NaN,Walter,Dimitri Schiltmans,<p><span>Inderdaad mooi exemplaar. Die witte h...,https://www.oervondstchecker.nl/images/large/5...


In [9]:
text_cols = data.select_dtypes(include="object").columns
print(text_cols)

Index(['date', 'name', 'description', 'type_code', 'category',
       'minimum period', 'maximum period', 'creator', 'reviewer',
       'reviewer_notes', 'images'],
      dtype='object')


In [10]:
INCLUDE = ['name', 'description','reviewer_notes']
text_cols = [c for c in text_cols if c in INCLUDE]
text_cols

['name', 'description', 'reviewer_notes']

In [2]:
import pandas as pd
from functools import lru_cache
from transformers import MarianMTModel, MarianTokenizer

model_name = "Helsinki-NLP/opus-mt-nl-en"
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)

@lru_cache(maxsize=200_000)
def translate_nl_en(text: str) -> str:
    text = str(text).strip()
    if not text:
        return text
    # tokenize + translate
    batch = tokenizer.prepare_seq2seq_batch([text], return_tensors="pt")
    out = model.generate(**batch)
    return tokenizer.batch_decode(out, skip_special_tokens=True)[0]

def translate_df_columns_free(data: pd.DataFrame, text_cols: list[str]) -> pd.DataFrame:
    # 1) stack unique strings
    s = data[text_cols].stack(dropna=False)
    mask = s.map(type).eq(str)
    uniq = pd.unique(s[mask])

    print(f"Unique strings to translate: {len(uniq):,}")

    # 2) build map
    trans_map = {u: translate_nl_en(u) for u in uniq}

    # 3) vectorized remap
    data[text_cols] = data[text_cols].apply(lambda col: col.map(lambda v: trans_map.get(v, v)))
    return data


/Users/georgianamg93gmail.com/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/georgianamg93gmail.com/miniconda3/lib/python3.13/site-packages/transformers/models/marian/tokenization_marian.py:175: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


In [11]:
data = translate_df_columns_free(data, text_cols)   # MarianMT version

/var/folders/_p/r6dt32s51pgc2j9lb94tb5qh0000gn/T/ipykernel_1785/1131702349.py:21: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  s = data[text_cols].stack(dropna=False)
/Users/georgianamg93gmail.com/miniconda3/lib/python3.13/site-packages/transformers/tokenization_utils_base.py:4291: FutureWarning: 
`prepare_seq2seq_batch` is deprecated and will be removed in version 5 of HuggingFace Transformers. Use the regular
`__call__` method to prepare your inputs and targets.

Here is a short example:

model_inputs = tokenizer(src_texts, text_target=tgt_texts, ...)

If you either need to use different keyword arguments for the source and target texts, you should do two calls like
this:

model_inputs = tokenizer(src_texts, ...)
labels = tokenizer(text_target=tgt_texts, ...)
model_inputs["l

Unique strings to translate: 47,353


In [12]:
data

,date,name,description,latitude,longitude,type_code,category,minimum period,maximum period,creator,reviewer,reviewer_notes,images
0,2023-01-12,Snippies dog/wolf?,Found today on the sand engine. Spot map doesn...,51.968006,3.973275,palaeontological,Mammals,n.C. 0 - v.C. 10.000 | Recent,n.C. 0 - v.C. 10.000 | Recent,Dimar,Charlie Schouwenburg,<p>No Maasvlakte</p> <p>This is a premier 4 fr...,https://www.oervondstchecker.nl/images/large/4...
1,2023-01-12,Round hollow bone fragment,"Not to determine, but where should I place suc...",51.948593,3.968725,palaeontological,Mammals,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 65.000 - v.C. 120.000 | Vroeg Weichselien,Geert,Hansjorg Ahrens,<p>A mid-hand or foot-bone shaft of a larger h...,https://www.oervondstchecker.nl/images/large/2...
2,2023-01-12,Piece of crown,Looks like a fragment of a crown Still determi...,51.945522,3.969669,palaeontological,Mammals,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 65.000 - v.C. 120.000 | Vroeg Weichselien,Geert,Charlie Schouwenburg,<p>That's a difficult one. My first impression...,https://www.oervondstchecker.nl/images/large/0...
3,2023-01-12,Quite complete bone?,I think it's a fairly complete bone. Can you t...,51.935310,3.977137,palaeontological,Mammals,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,Geert,Charlie Schouwenburg,"<p>This is a phrase of the third toe koot, aro...",https://www.oervondstchecker.nl/images/large/3...
4,2023-01-12,Bone fragment,Unfortunately quite damaged but perhaps by ide...,51.928749,3.980312,palaeontological,Mammals,n.C. 0 - v.C. 10.000 | Recent,n.C. 0 - v.C. 10.000 | Recent,Geert@gmail.com,Charlie Schouwenburg,<p>Despite all the forms I can't determine it ...,https://www.oervondstchecker.nl/images/large/1...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19834,2014-01-24,Strings?,Possibly sharing antlers?,51.960930,3.961022,palaeontological,Mammals,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,v.C. 120.000 - v.C. 140.000 | Midden Pleistoce...,Johan Grootveld,Charlie Schouwenburg,<p>I don't think of an antler in all three cas...,https://www.oervondstchecker.nl/images/large/5...
19835,2014-01-24,Harpoon Point,Looks like a harpoon point.,51.939095,3.972733,archaeological,Tools,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 5.000 - v.C. 15.000 | Holoceen,Walter,Dimitri Schiltmans,<p><span>Another fine specimen of bone-cut har...,https://www.oervondstchecker.nl/images/large/5...
19836,2014-01-24,Skull fragment,Unique find of a skull fragment,51.976078,3.972673,archaeological,Human,v.C. 5.000 - v.C. 15.000 | Holoceen,v.C. 5.000 - v.C. 15.000 | Holoceen,Walter,Hansjorg Ahrens,<p>This is a part of the right half of a human...,https://www.oervondstchecker.nl/images/large/5...
19837,2014-01-24,White shark's tooth,This morning another top finding made on the M...,51.956669,3.962030,palaeontological,NaN,NaN,NaN,Walter,Dimitri Schiltmans,<p><span>Indeed beautiful specimen. That white...,https://www.oervondstchecker.nl/images/large/5...


In [13]:
output = '2023-01-13Data-exportOervondstchecker_translated.parquet'
data.to_parquet(output)